# Reproduce ROCKET on ECG200

---

### 1. Imports

In [3]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import RidgeClassifierCV
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

from sktime.datasets import load_UCR_UEA_dataset
from sktime.transformations.panel.rocket import Rocket

nb_dir = Path.cwd()
PROJECT_ROOT = nb_dir.parents[1]
SRC_ROOT = PROJECT_ROOT / "src"

if str(SRC_ROOT) not in sys.path:
    sys.path.append(str(SRC_ROOT))

In [4]:
import warnings
warnings.filterwarnings("ignore", module="sktime")

import os
os.environ["OMP_DISPLAY_ENV"] = "FALSE"

### 2. Load ECG200 via sktime

In [5]:
X_train_df, y_train = load_UCR_UEA_dataset(
    name="ECG200",        
    split="train",
    return_X_y=True
)
X_test_df, y_test = load_UCR_UEA_dataset(
    name="ECG200",       
    split="test",
    return_X_y=True
)

### 3. Convert sktime panel format to NumPy arrays

In [6]:
# ItalyPowerDemand is univariate: one column with Series per row
X_train = np.vstack(X_train_df.iloc[:, 0].apply(lambda s: s.to_numpy()).to_numpy())
X_test = np.vstack(X_test_df.iloc[:, 0].apply(lambda s: s.to_numpy()).to_numpy())

y_train = np.array(y_train)
y_test = np.array(y_test)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((100, 96), (100, 96), (100,), (100,))

### 4. Normalise data and apply ROCKET (sktime)

In [7]:
# z-normalise per series
scaler = StandardScaler(with_mean=True, with_std=True)
X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.transform(X_test)

def to_panel(X):
    return pd.DataFrame({0: [pd.Series(row) for row in X]})

X_train_panel = to_panel(X_train_norm)
X_test_panel = to_panel(X_test_norm)

rocket = Rocket(num_kernels=10000, random_state=42)
rocket.fit(X_train_panel)
X_train_features = rocket.transform(X_train_panel)
X_test_features = rocket.transform(X_test_panel)

X_train_features.shape, X_test_features.shape

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


((100, 20000), (100, 20000))

### 5. Train ridge classifier and evaluate

In [8]:
clf = RidgeClassifierCV(alphas=np.logspace(-3, 3, 7))
clf.fit(X_train_features, y_train)
y_pred = clf.predict(X_test_features)

acc = accuracy_score(y_test, y_pred)
acc

0.89

### 6. Save results to CSV for later comparison

In [9]:
import csv
from pathlib import Path

RESULTS_DIR = Path(PROJECT_ROOT) / "experiments" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

out_path = RESULTS_DIR / "ecg200_rocket_sktime.csv"

with out_path.open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["dataset", "n_kernels", "train_size", "test_size", "accuracy"])
    writer.writerow(["ECG200", 10000, X_train.shape[0], X_test.shape[0], acc])

print("Saved results to:", out_path.relative_to(PROJECT_ROOT))

Saved results to: experiments/results/ecg200_rocket_sktime.csv
